# S-DoT Paper Reproduction01 - merge

In [1]:
# matplotlib 설정
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

# 6개년 데이터 하나의 스키마로 통합
- 통합하는 과정에서 파일 별로 스키마가 다른 것을 확인했음
- 컬럼명에 오타도 존재함 (ex. 흑구 운도)
- minmax 데이터 없는 측정항목도 존재함
- 진동의 단위가 g으로 다름 (다른 파일에서는 mm/s)

In [2]:
import pandas as pd

test_file = "../data/S-DoT_2021/S-DoT_NATURE_2021.10.25-10.31.csv"

df_test = pd.read_csv(test_file, encoding='cp949', low_memory=False)
print(df_test.columns.tolist())
print(repr(df_test.columns[0]))  # 첫 컬럼명을 repr로 찍어서 숨은 공백/특수문자 확인

['기관 명', '모델명', '시리얼', '구분', '기온(℃)', '상대습도( %)', '풍향(°)', '풍속(m/s)', '돌풍 풍향(°)', '돌풍 풍속(m/s)', '조도(lux)', '자외선(UVI)', '소음(dB)', '진동_x(g)', '진동_y(g)', '진동_z(g)', '진동_x 최대(g)', '진동_y 최대(g)', '진동_z 최대(g)', '흑구 운도(℃)', '전송시간', '등록일자']
'기관 명'


### 통합 전 스키마 확인

In [3]:
import glob
import pandas as pd

files = glob.glob("../data/**/*.csv", recursive=True)

schema_groups = {}
for f in files:
    try:
        cols = tuple(pd.read_csv(f, encoding='cp949', nrows=0).columns.tolist())
    except:
        try:
            cols = tuple(pd.read_csv(f, encoding='utf-8-sig', nrows=0).columns.tolist())
        except Exception as e:
            print(f"읽기 실패: {f} - {e}")
            continue
    schema_groups.setdefault(cols, []).append(f)

print(f"총 {len(schema_groups)}개의 서로 다른 스키마 발견")
for i, (cols, flist) in enumerate(schema_groups.items()):
    print(f"\n--- 스키마 {i+1} ({len(flist)}개 파일) ---")
    print(cols[:8], '...')
    print("예시 파일:", flist[0])

총 14개의 서로 다른 스키마 발견

--- 스키마 1 (303개 파일) ---
('모델번호', '시리얼', '측정시간', '지역', '자치구', '행정동', '온도 최대(℃)', '온도 평균(℃)') ...
예시 파일: ../data/S-DoT_NATURE_2026.06.22-06.28.csv

--- 스키마 2 (1개 파일) ---
('시리얼_matched', '위도', '경도') ...
예시 파일: ../data/processed_sensor_coords.csv

--- 스키마 3 (1개 파일) ---
('기관 명', '모델명', '시리얼', '구분', '기온(℃)', '상대습도( %)', '풍향(°)', '풍속(m/s)') ...
예시 파일: ../data/S-DoT_2020/S-DoT_NATURE_2020.11.01-11.08.csv

--- 스키마 4 (105개 파일) ---
('기관 명', '모델명', '시리얼', '구분', '기온(℃)', '상대습도( %)', '풍향(°)', '풍속(m/s)') ...
예시 파일: ../data/S-DoT_2020/S-DoT_NATURE_2020.11.23-11.29.csv

--- 스키마 5 (1개 파일) ---
('기관명', '모델명', '시리얼', '구분', '기온 (℃)', '상대습도 (%)', '풍향 (°)', '풍속 (m/s)') ...
예시 파일: ../data/S-DoT_2020/S-DoT_NATURE_2020.09.01-09.30.csv

--- 스키마 6 (5개 파일) ---
('기관 명', '모델명', '시리얼', '구분', ' 기온(℃) ', '상대습도( %)', '풍향(°)', '풍속(m/s)') ...
예시 파일: ../data/S-DoT_2020/S-DoT_NATURE_2020.06.01-06.30.csv

--- 스키마 7 (1개 파일) ---
('기관 명', '모델명', '시리얼', '구분', '기온(℃)', '상대습도( %)', '풍향(°)', '풍속(m/s)') ...
예시 파일

### 컬럼명 정규화 -> pure group 확인

In [4]:
import re

def normalize_col(col):
    col = col.strip()  # 앞뒤 공백 제거
    col = re.sub(r'\s+', '', col)  # 중간 공백도 전부 제거 (괄호 앞 공백 등)
    return col

def get_normalized_schema(cols):
    return tuple(sorted(normalize_col(c) for c in cols))  # sorted로 순서 차이도 무시

schema_groups_normalized = {}
for f in files:
    try:
        cols = pd.read_csv(f, encoding='cp949', nrows=0).columns.tolist()
    except:
        cols = pd.read_csv(f, encoding='utf-8-sig', nrows=0).columns.tolist()
    norm_schema = get_normalized_schema(cols)
    schema_groups_normalized.setdefault(norm_schema, []).append(f)

print(f"정규화 후 {len(schema_groups_normalized)}개 스키마 그룹")
for i, (cols, flist) in enumerate(schema_groups_normalized.items()):
    print(f"그룹 {i+1}: {len(flist)}개 파일")

정규화 후 12개 스키마 그룹
그룹 1: 303개 파일
그룹 2: 1개 파일
그룹 3: 6개 파일
그룹 4: 105개 파일
그룹 5: 2개 파일
그룹 6: 1개 파일
그룹 7: 2개 파일
그룹 8: 5개 파일
그룹 9: 1개 파일
그룹 10: 1개 파일
그룹 11: 1개 파일
그룹 12: 1개 파일


In [5]:
# 각 정규화된 그룹의 파일들에서 날짜 범위 확인
import re

for i, (cols, flist) in enumerate(schema_groups_normalized.items()):
    dates = []
    for f in flist:
        m = re.search(r'(\d{4})[._](\d{2})[._](\d{2})', f)
        if m:
            dates.append(f"{m.group(1)}-{m.group(2)}")
    dates = sorted(set(dates))
    print(f"그룹 {i+1}: {len(flist)}개 파일, 날짜범위 {dates[0] if dates else '?'} ~ {dates[-1] if dates else '?'}")

그룹 1: 303개 파일, 날짜범위 2023-01 ~ 2026-06
그룹 2: 1개 파일, 날짜범위 ? ~ ?
그룹 3: 6개 파일, 날짜범위 2020-04 ~ 2020-11
그룹 4: 105개 파일, 날짜범위 2020-11 ~ 2022-12
그룹 5: 2개 파일, 날짜범위 2020-09 ~ 2020-10
그룹 6: 1개 파일, 날짜범위 2020-11 ~ 2020-11
그룹 7: 2개 파일, 날짜범위 2021-11 ~ 2022-12
그룹 8: 5개 파일, 날짜범위 2021-01 ~ 2021-04
그룹 9: 1개 파일, 날짜범위 2021-09 ~ 2021-09
그룹 10: 1개 파일, 날짜범위 2022-12 ~ 2022-12
그룹 11: 1개 파일, 날짜범위 2022-12 ~ 2022-12
그룹 12: 1개 파일, 날짜범위 2022-02 ~ 2022-02


### 스키마 확인 결과
- 크게 구형(~2022년)과 신형(2023년~)으로 나뉨
- 구형 스키마의 경우 min/avg/max 구조 없음, 단위도 다름, 진동 등 일부 변수 자체가 아예 없음
- 동일 측정항목인데 컬럼명이 다른 경우도 있음 (ex. 신형 스키마에서는 `온도 최대(℃), 온도 평균(℃), 온도 최소(℃)`, 구형 스키마에서는 `기온(℃)`)

# 2023년 이후 신형 스키마 통합
- 구형 -> 신형 컬럼 매핑해서 5개년 억지로 통합하게 되면 단위 변환(g→mm/s 등)이 물리적으로 불가능한 변수가 있어서 데이터 신뢰성이 떨어짐
- 따라서 2023년~2025년 신형 스키마 데이터만 통합하기로 결정

In [6]:
# 신형 스키마(그룹 1) 파일만 필터링해서 로드
new_schema_files = schema_groups_normalized[list(schema_groups_normalized.keys())[0]]  # 그룹1 키 확인 필요

# 2023-2025년 6~9월만 필터
import re

def extract_year_month(filepath):
    m = re.search(r'(\d{4})[._](\d{2})', filepath)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None

target_files = []
for f in new_schema_files:
    year, month = extract_year_month(f)
    if year in [2023, 2024, 2025] and month in [6,7,8,9]:
        target_files.append(f)

print(f"재현 대상 파일: {len(target_files)}개")

재현 대상 파일: 88개


In [7]:
import os
import glob
import re
import pandas as pd

### 2023-2025년 6-9월 파일만 필터링

In [8]:
# 신형 스키마 그룹 찾기 (컬럼 개수가 가장 많고 '측정시간'이 포함된 그룹)
new_schema_key = None
for cols in schema_groups_normalized:
    if '측정시간' in cols and len(cols) > 20:  # 51개 컬럼짜리 신형 스키마 특징
        new_schema_key = cols
        break

new_schema_files = schema_groups_normalized[new_schema_key]
print(f"신형 스키마 파일 수: {len(new_schema_files)}")

def extract_year_month(filepath):
    m = re.search(r'(\d{4})[._](\d{2})', filepath)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None

target_files = []
for f in new_schema_files:
    year, month = extract_year_month(f)
    if year in [2023, 2024, 2025] and month in [6, 7, 8, 9]:
        target_files.append(f)

print(f"재현 대상 파일: {len(target_files)}개")

신형 스키마 파일 수: 303
재현 대상 파일: 88개


### 파일 로드 + 병합

In [9]:
def load_target_files(target_files):
    dfs = []
    for f in target_files:
        try:
            df = pd.read_csv(f, encoding='cp949', low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(f, encoding='utf-8-sig', low_memory=False)
        df['__source_file'] = f
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

repro_df = load_target_files(target_files)
print("병합 완료:", repro_df.shape)

병합 완료: (5439903, 59)


### 측정시간 파싱

In [10]:
# 원본 백업
repro_df['측정시간_raw'] = repro_df['측정시간']

# 1차: 언더스코어 포맷
repro_df['측정시간'] = pd.to_datetime(
    repro_df['측정시간_raw'], format='%Y-%m-%d_%H:%M:%S', errors='coerce'
)

# 2차: 실패한 것만 표준 포맷 재시도
mask_failed = repro_df['측정시간'].isna()
if mask_failed.sum() > 0:
    repro_df.loc[mask_failed, '측정시간'] = pd.to_datetime(
        repro_df.loc[mask_failed, '측정시간_raw'], errors='coerce'
    )

n_failed = repro_df['측정시간'].isna().sum()
print(f"파싱 실패: {n_failed}개 / {len(repro_df)}개 ({n_failed/len(repro_df)*100:.2f}%)")

파싱 실패: 128619개 / 5439903개 (2.36%)


### 결측 제거 + 이중 기간 필터링 (연/월 재확인)

In [11]:
repro_df = repro_df.dropna(subset=['측정시간'])

repro_df = repro_df[
    repro_df['측정시간'].dt.year.isin([2023, 2024, 2025]) &
    repro_df['측정시간'].dt.month.isin([6, 7, 8, 9])
]

print("최종 shape:", repro_df.shape)
print(repro_df['측정시간'].min(), '~', repro_df['측정시간'].max())
print("고유 센서 수:", repro_df['시리얼'].nunique())

최종 shape: (4459253, 60)
2023-06-05 01:07:00 ~ 2025-09-30 23:07:00
고유 센서 수: 1159


In [12]:
# 6월 초 데이터 있는지 확인
repro_df[repro_df['측정시간'].dt.strftime('%Y-%m') == '2023-06']['측정시간'].dt.day.value_counts().sort_index()

측정시간
5     22439
6     22502
7     22877
8     22810
9     22379
10    22481
11    21238
12    21979
13    22242
14    22050
15    22211
16    21827
17    22043
18    24675
19    24754
20    22286
21    24340
22    24436
23    22213
24    21934
25    20164
26    22085
27    22688
28    22126
29    21994
30    23005
Name: count, dtype: int64

# 위경도 정보 merge

In [13]:
loc_df = pd.read_excel('../data/서울시 도시데이터 센서(S-DoT) 환경정보 설치 위치정보.xlsx')  
print(loc_df.columns.tolist())
print(loc_df.shape)
print(loc_df.head(3))

['No', '모델 시리얼(*)', '주소', '좌표 구분코드', '위도', '경도', '변경 전 시리얼', '변경 전 시리얼(데이터 상 표기)']
(1170, 8)
   No    모델 시리얼(*)                 주소 좌표 구분코드         위도          경도 변경 전 시리얼  \
0   1  V02Q1940655  서울특별시 종로구 북촌로6길 1     W84  37.580051  126.985146      NaN   
1   2  V02Q1940539  서울특별시 종로구 이화장길 33     W84  37.576920  127.004514      NaN   
2   3  V02Q1940737  서울특별시 종로구 평창10길 5     W84  37.605426  126.967435      NaN   

  변경 전 시리얼(데이터 상 표기)  
0                NaN  
1                NaN  
2                NaN  


In [14]:
# 조인 키 컬럼명 통합
loc_df = loc_df.rename(columns={'모델 시리얼(*)': '시리얼'})

In [15]:
# 조인 매칭률 미리 확인
sensors_in_data = set(repro_df['시리얼'].unique())
sensors_in_loc = set(loc_df['시리얼'].unique())

print(f"repro_df 고유 센서: {len(sensors_in_data)}개")
print(f"loc_df 고유 센서: {len(sensors_in_loc)}개")
print(f"매칭되는 센서: {len(sensors_in_data & sensors_in_loc)}개")
print(f"repro_df에만 있고 loc_df엔 없는 센서: {len(sensors_in_data - sensors_in_loc)}개")

repro_df 고유 센서: 1159개
loc_df 고유 센서: 1170개
매칭되는 센서: 1155개
repro_df에만 있고 loc_df엔 없는 센서: 4개


In [16]:
# 변경 전 시리얼로 2차 매칭
unmatched = sensors_in_data - sensors_in_loc
print(f"1차 미매칭: {len(unmatched)}개")

# 변경 전 시리얼 컬럼들로 보조 매핑 테이블 만들기
old_serial_map = {}
for _, row in loc_df.iterrows():
    if pd.notna(row['변경 전 시리얼']):
        old_serial_map[row['변경 전 시리얼']] = row['시리얼']
    if pd.notna(row['변경 전 시리얼(데이터 상 표기)']):
        old_serial_map[row['변경 전 시리얼(데이터 상 표기)']] = row['시리얼']

# repro_df의 시리얼을 신형으로 치환
repro_df['시리얼_matched'] = repro_df['시리얼'].map(old_serial_map).fillna(repro_df['시리얼'])

# 재확인
sensors_after_fix = set(repro_df['시리얼_matched'].unique())
print(f"보정 후 매칭: {len(sensors_after_fix & sensors_in_loc)}개")

1차 미매칭: 4개
보정 후 매칭: 1154개


In [17]:
still_unmatched = sensors_after_fix - sensors_in_loc
print(still_unmatched)

# 이 4개 센서가 repro_df에서 몇 개 행을 차지하는지 (영향도 확인)
affected_rows = repro_df[repro_df['시리얼_matched'].isin(still_unmatched)]
print(f"영향받는 행 수: {len(affected_rows)}개 / 전체 {len(repro_df)}개")

set()
영향받는 행 수: 0개 / 전체 4459253개


In [18]:
merge_key = '시리얼_matched' if '시리얼_matched' in repro_df.columns else '시리얼'

merged_df = repro_df.merge(
    loc_df[['시리얼', '위도', '경도', '주소']],
    left_on=merge_key,
    right_on='시리얼',
    how='left',
    suffixes=('', '_loc')
)

print("병합 후 shape:", merged_df.shape)
print("위경도 결측 센서 수:", merged_df[merged_df['위도'].isna()]['시리얼_x' if '시리얼_x' in merged_df.columns else '시리얼'].nunique())

# 위경도 없는 행 제거 (Haversine 계산에 필요)
merged_df = merged_df.dropna(subset=['위도', '경도'])
print("최종 shape (위경도 있는 것만):", merged_df.shape)
print("최종 고유 센서 수:", merged_df[merge_key].nunique())

병합 후 shape: (4459253, 65)
위경도 결측 센서 수: 0
최종 shape (위경도 있는 것만): (4459253, 65)
최종 고유 센서 수: 1154


In [19]:
sensor_coords = merged_df[[merge_key, '위도', '경도']].drop_duplicates(subset=[merge_key]).reset_index(drop=True)
print(sensor_coords.shape)
print(sensor_coords.head())

sensor_coords.to_csv('../data/processed_sensor_coords.csv', index=False, encoding='utf-8-sig')
print("저장 완료: ../data/processed_sensor_coords.csv")

(1154, 3)
   시리얼_matched         위도          경도
0  OC3CL200304  37.495775  126.954450
1  OC3CL200305  37.504948  126.938966
2  OC3CL200012  37.544002  127.069731
3  OC3CL200013  37.583469  126.982622
4  OC3CL200014  37.563574  126.984504
저장 완료: ../data/processed_sensor_coords.csv


In [23]:
merged_df.to_csv('../data/processed_merged_df.csv', index=False, encoding='utf-8-sig')
print("저장 완료:", merged_df.shape)

저장 완료: (4459253, 65)
